# Phase 7 — Human-in-the-Loop (approval gates)

**A light phase.** One small idea, one small TODO.

**Goal**: So far the agent could call its tools freely. But in a real system some actions are expensive, risky, or irreversible — you want a **gate**: something that checks each tool call *before* it runs and can **deny** it.

The Claude Agent SDK gives you exactly one hook for this: the **`can_use_tool`** callback. Every time the agent wants to use a tool, the SDK calls your function first. You return **allow** or **deny**.

In a real product, "deny" is where a *human* clicks a button. Here we automate the decision with a **rule** — but it's the same mechanism.

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Approval gate** | A checkpoint every tool call must pass before it runs. |
| **`can_use_tool`** | The SDK callback that *is* the gate. Returns allow / deny per call. |
| **`PermissionResultAllow` / `PermissionResultDeny`** | The two things your gate can return. |
| **Human-in-the-loop** | A human (or a policy standing in for one) approves risky actions. |

## Acceptance criteria
1. The `can_use_tool` gate is consulted on every tool call
2. A normal-sized query is **allowed** and answered correctly
3. An oversized query is **denied** by the gate
4. `Trace` exported to `traces/traces.jsonl`

## 1. Setup

In [ ]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

In [ ]:
import pandas as pd

from orchestrator.tools import get_retail_df
from orchestrator import tools
from orchestrator.observability import Trace, Timer
from orchestrator.agents import run_agent
from claude_agent_sdk import (
    tool, create_sdk_mcp_server, ClaudeAgentOptions,
    PermissionResultAllow, PermissionResultDeny,
)

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")

df = get_retail_df()
print(f"Loaded {len(df):,} rows")
print(f"Dev model: {DEV_MODEL}")

## 2. Ground truth

A normal question to test the allowed path: top 5 countries by revenue in 2011.

In [ ]:
df_2011 = df[(df["Quantity"] > 0) & (df["InvoiceDate"].dt.year == 2011)]

ground_truth = (
    df_2011.groupby("Country")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
print("Top 5 countries by revenue, 2011:")
print(ground_truth)

expected_countries = sorted(ground_truth["Country"])
print()
print("Expected countries in the answer:", expected_countries)

## 3. Rebuild the retail tool + MCP server (recap)

Same `query_retail` tool. Just run it.

In [ ]:
@tool(
    "query_retail",
    description=(
        "Query the retail transactions dataset to return the top N entries "
        "ranked by a metric. Arguments: year (optional), country (optional - "
        "OMIT to include all countries), top_n (default 10), group_by (one of "
        "'StockCode', 'Country', 'Customer ID'), metric (one of 'revenue', "
        "'Quantity'). Returns a list of dicts, one row each."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "year":     {"type": "integer", "description": "Calendar year filter, e.g. 2011"},
            "country":  {"type": "string",  "description": "Optional country filter. OMIT to include all countries."},
            "top_n":    {"type": "integer", "description": "How many rows to return"},
            "group_by": {"type": "string",  "description": "'StockCode', 'Country', or 'Customer ID'"},
            "metric":   {"type": "string",  "description": "'revenue' or 'Quantity'"},
        },
        "required": [],
    },
)
async def query_retail_tool(args):
    """Adapter: the agent passes args as a dict; we call the pandas function."""
    rows = tools.query_retail(
        year=args.get("year"),
        country=args.get("country"),
        top_n=args.get("top_n", 10),
        group_by=args.get("group_by", "StockCode"),
        metric=args.get("metric", "revenue"),
    )
    import json
    return {"content": [{"type": "text", "text": json.dumps(rows, default=str)}]}


retail_server = create_sdk_mcp_server(name="retail", version="1.0.0", tools=[query_retail_tool])
print("[OK] Retail tool + MCP server ready")

## 4. The approval gate

`can_use_tool` is an **async function** the SDK calls before every tool use. It receives the tool name and the arguments the agent wants to use, and it returns:

- `PermissionResultAllow()` — let the call run
- `PermissionResultDeny(message="...")` — block it; the message tells the agent why

Our rule: **queries with `top_n > 100` are too expensive — deny them.** Everything else is allowed. (In a real app, this is where a human would click Approve / Deny.)

**TODO 1**: write the approval rule.

In [ ]:
gate_log = []   # records every decision the gate makes, so we can inspect it


async def approval_gate(tool_name, tool_input, context):
    """The human-in-the-loop approval gate.

    The SDK calls this before every tool use. Return Allow to let it run,
    or Deny to block it.
    """
    top_n = tool_input.get("top_n", 10)

    # ----- TODO 1: write the approval rule --------------------------------
    if top_n > 100:
        gate_log.append(("DENY", tool_name, top_n))
        return PermissionResultDeny(
            message=f"top_n={top_n} exceeds the 100-row limit — blocked by the approval gate"
        )
    gate_log.append(("ALLOW", tool_name, top_n))
    return PermissionResultAllow()
    # ----------------------------------------------------------------------


print("[OK] approval_gate defined")

## 5. The agent — gated by `can_use_tool`

We pass `can_use_tool=approval_gate` into the options. Note there is **no `allowed_tools`** here — that's deliberate. Without a static allow-list, *every* tool call must pass through your gate.

In [ ]:
gated_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=(
        "You are a retail data analyst. Answer using the query_retail tool "
        "and present results as a markdown table. Never invent numbers. "
        "If a tool call is blocked by the approval gate, tell the user "
        "plainly that the request was denied and why."
    ),
    mcp_servers={"retail": retail_server},
    can_use_tool=approval_gate,    # <-- the gate
    max_turns=5,
)
print("[OK] Gated agent configured")

## 6. Demo — a request the gate ALLOWS

A normal question. `top_n` is small, so the gate allows the call and the agent answers.

In [ ]:
gate_log.clear()
allow_run = await run_agent(
    "DataAgent",
    "What were the top 3 products by revenue in 2011?",
    gated_options,
)
print("----- Agent answer -----")
print(allow_run.answer)
print()
print("Gate decisions:", gate_log)

## 7. Demo — a request the gate DENIES

Now we ask for an absurdly large result (top **5000**). The agent calls the tool with a big `top_n`, the gate **denies** it, and the agent has to report that — or retry smaller.

In [ ]:
gate_log.clear()
deny_run = await run_agent(
    "DataAgent",
    "List the top 5000 products by revenue in 2011 - I want the full ranking.",
    gated_options,
)
print("----- Agent answer -----")
print(deny_run.answer)
print()
print("Gate decisions:", gate_log)
print()
denied = [d for d in gate_log if d[0] == "DENY"]
print(f"The gate denied {len(denied)} call(s).")

## 8. Run + verify

**TODO 2**: write a normal question (top 5 countries by revenue in 2011) — small enough that the gate allows it.

In [ ]:
# ----- TODO 2: write your question -----
QUESTION = "What were the top 5 countries by revenue in 2011?"
# ---------------------------------------

gate_log.clear()
trace = Trace(model=DEV_MODEL, question=QUESTION)
with Timer() as t:
    run = await run_agent("DataAgent", QUESTION, gated_options)

trace.input_tokens        = run.input_tokens
trace.output_tokens       = run.output_tokens
trace.cached_input_tokens = run.cached_input_tokens
trace.n_tool_calls        = run.n_tool_calls
trace.n_delegations       = 1
trace.latency_ms          = t.elapsed_ms
trace.answer              = run.answer
trace.compute_cost()

print("----- Agent answer -----")
print(run.answer)
print()
print("Gate decisions for this run:", gate_log)

# ----- TODO 3: write the assertion -----
missing = []
for country in expected_countries:
    if country not in trace.answer:
        missing.append(country)
# ---------------------------------------

trace.passed = (missing == [])
print(f"\nMissing countries: {missing}")
print(f"Passed: {trace.passed}")
assert trace.passed, f"Answer missed: {missing}"

# Phase 7 acceptance: the gate was consulted and allowed this normal query
assert len(gate_log) >= 1, "Expected the approval gate to be consulted at least once"
assert any(d[0] == "ALLOW" for d in gate_log), "Expected the gate to ALLOW this normal query"

trace.append_jsonl()
print("\n[OK] Acceptance criteria met - gate consulted, normal query allowed, trace saved")

## 9. Quiz (~5 min — answer in a new markdown cell)

**TODO 4**: Answer in your own words.

1. **Why a gate**: agents can call tools on their own. Why would you want a checkpoint that can *deny* a tool call? Give one concrete example of an action you'd want a human to approve first.

   Because "the model thinks it's a good idea" isn't a guarantee — agents can pick the wrong tool, pass dangerous args, or string together actions whose combined effect is destructive. **Example**: an agent with a `send_email` tool composing an exec-team-wide "Q1 revenue down 30%" message based on a query it ran on partial data. You want a human to look at the draft *before* it leaves the org. Same logic for `delete_records`, `wire_funds`, `deploy_to_prod` — anything irreversible or visible to others.

2. **`can_use_tool`**: when exactly does the SDK call your `can_use_tool` function — before or after the tool runs? Why does the timing matter?

   **Before**, every single time. The agent emits a `tool_use` block → SDK pauses → calls your gate → only if you return `Allow` does the tool actually execute. Timing matters because a post-hoc check is useless for irreversible actions — once the email is sent or the row is deleted, "the gate said no" is too late. Pre-execution is the only point where "deny" is meaningful.

3. **No `allowed_tools`**: this phase removed the static `allowed_tools` list. What's the difference between a static allow-list and the dynamic `can_use_tool` gate? When would you want each?

   **Static allow-list** answers "which tools may the agent *ever* use?" — a coarse boolean per tool name, set at config time. **Dynamic gate** answers "may the agent use this tool *with these specific arguments right now*?" — fine-grained, runtime-aware, can look at the call's payload. Use both together: the allow-list bars whole categories (no `delete_records` ever); the gate adjudicates the calls that pass the list (allow `send_email` to internal addresses, deny to external ones, ask a human for boundary cases).

4. **Automating the human**: here the "human" is a rule (`top_n > 100`). In a real product, what would replace that rule — and what's the trade-off of a human gate vs an automatic rule (think speed vs judgment)?

   Replace it with a UI prompt — Slack DM, web modal, mobile push — showing the tool name, the args, and Approve/Deny buttons (Claude Desktop's permission popup is exactly this). **Trade-off**: rules are *instant* and consistent but only as smart as the rule-writer — they over-block on novel edge cases ("a 101-row export was actually fine here") or under-block on adversarial inputs that fit the rule but feel wrong. Humans bring *judgment* but cost seconds-to-minutes per call and don't scale. The right answer is usually a **tiered policy**: auto-allow obvious safe calls, auto-deny obvious bad ones, escalate the ambiguous middle to a human.

## Phase 7 done when:
- [x] TODO 1 (the approval rule) filled in
- [x] TODO 2 (your question) filled in
- [x] TODO 3 (assertion) filled in
- [x] TODO 4 (quiz) answered
- [x] Cell 13 shows a request being ALLOWED
- [x] Cell 15 shows a request being DENIED
- [x] Notebook runs top-to-bottom without errors
- [x] `traces/traces.jsonl` has a new line

Then ping me — we'll review, then move to Phase 8 (guardrails).